In [ ]:
import os, sys, json, time, re, importlib, hashlib
import numpy as np
import pandas as pd
import torch

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")

def find_phase9():
    for d in [os.environ.get("MELD_NOTEBOOK_DIR", ""), BASE_DIR, os.getcwd(), "/kaggle/working"]:
        if d and os.path.isfile(os.path.join(d, "Phase_09_Model_Training.ipynb")):
            return os.path.join(d, "Phase_09_Model_Training.ipynb")
    raise FileNotFoundError("Phase_09_Model_Training.ipynb not found (set MELD_NOTEBOOK_DIR). Use the STEP C version.")

P9 = ["".join(c["source"]) for c in json.load(open(find_phase9()))["cells"] if c["cell_type"] == "code"]
_all = "\n".join(P9)
assert all(k in _all for k in ("MELD_P9_LR", "MELD_P9_UTT_MASK_P", "MELD_P9_SWEEP_TAG")), "This Phase_09 notebook is older than STEP C -- use the STEP C version."
_set = [k for k in ("MELD_P9_LR", "MELD_P9_WD", "MELD_P9_DROPOUT", "MELD_P9_LAMBDAS", "MELD_P9_SEEDS") if os.environ.get(k)]
assert ROBUST_RHO > 0, "Tuning selects by the robust score: MELD_P9_ROBUST_RHO must be > 0 (default 0.5)."
assert not _set, f"Unset {_set} before tuning: the tuner must start from Phase 9's DEFAULT recipe (it varies these itself)."

def _cell(marker):
    hits = [c for c in P9 if marker in c]
    assert len(hits) == 1, f"Phase 9 cell with marker {marker!r} found {len(hits)} times"
    return hits[0]

for marker in ("# 1. Configuration (requires Phases 1-8)", "MODULE_SRC = r'''", "# 3. Data (train / dev only)"):
    exec(compile(_cell(marker), "<phase9:" + marker[:20] + ">", "exec"), globals())       

_ex = _cell("# STEP 3: run folders")                                               
exec(compile(_ex[_ex.index("import hashlib\nFEATURE_DESC"):_ex.index("\n", _ex.index('print(f"Run folder:'))], "<phase9:folder-tag>", "exec"), globals())
TUNE_DIR = os.path.join(CKPT_DIR, os.path.basename(PHASE9_DIR).replace("phase9", "tuning", 1))
TUNE_RES = os.path.join(RESULTS_DIR, "tuning"); os.makedirs(TUNE_RES, exist_ok=True); os.makedirs(TUNE_DIR, exist_ok=True)
print(f"\nTuning run folder: {TUNE_DIR}\nTuning results   : {TUNE_RES}\nBase recipe HP   : {HP}")

In [ ]:

def _envf(k, d): return float(os.environ.get(k, d))
def _envl(k, d, cast): return [cast(x) for x in os.environ.get(k, d).split(",") if x.strip()]
N_RANDOM      = int(os.environ.get("MELD_TUNE_N", 16))              
TOP_K         = int(os.environ.get("MELD_TUNE_K", 3))               
TUNE_SEED     = int(os.environ.get("MELD_TUNE_SEED", 2026))         
SCREEN_SEED   = int(os.environ.get("MELD_TUNE_SCREEN_SEED", 42))
FRESH_SEEDS   = _envl("MELD_TUNE_FRESH_SEEDS", "43,44,45", int)     
LAMBDA_SCREEN = _envf("MELD_TUNE_LAMBDA_SCREEN", 2.0)              
LAMBDA_GRID   = _envl("MELD_TUNE_LAMBDAS", "0.5,1,2,4,8", float)
MIN_GAIN      = _envf("MELD_TUNE_MIN_GAIN", 0.01)
MAX_HOURS     = _envf("MELD_TUNE_MAX_HOURS", 0)                     
SPACE = {"lr": _envl("MELD_TUNE_LR_RANGE", "1e-4,1e-3", float), "wd": _envl("MELD_TUNE_WD_RANGE", "1e-3,1e-1", float),
         "dropout": _envl("MELD_TUNE_DROPOUTS", "0.2,0.3,0.4,0.5", float)}
assert SCREEN_SEED not in FRESH_SEEDS, "the decision seeds must exclude the screening seed"

def make_id(lr, wd, dropout):
    return f"lr{lr:.3g}_wd{wd:.2g}_do{dropout:g}"

def sample_candidates(n, seed, default, space):
    rs = np.random.RandomState(seed)
    drops = sorted(set(space["dropout"]) | {float(default["dropout"])})
    cands = [{**default, "id": make_id(default["lr"], default["wd"], default["dropout"]), "tag": "default"}]
    seen = {cands[0]["id"]}
    while len(cands) < n + 1:
        lr = float(f"{np.exp(rs.uniform(np.log(space['lr'][0]), np.log(space['lr'][1]))):.3g}")
        wd = float(f"{np.exp(rs.uniform(np.log(space['wd'][0]), np.log(space['wd'][1]))):.2g}")
        do = float(rs.choice(drops))
        cid = make_id(lr, wd, do)
        if cid not in seen:
            seen.add(cid); cands.append({"id": cid, "lr": lr, "wd": wd, "dropout": do, "tag": "random"})
    return cands

class Budget:
    def __init__(self, max_hours, clock=time.time):
        self.t0, self.max_s, self.clock = clock(), max_hours * 3600.0, clock
    def exceeded(self):
        return self.max_s > 0 and (self.clock() - self.t0) >= self.max_s
    def elapsed_h(self):
        return (self.clock() - self.t0) / 3600.0

def run_key(cid, model, lam, seed):
    return (cid, model, round(float(lam), 6), int(seed))

def execute(plan, run_fn, RUNS, budget, log=print):
    todo = [p for p in plan if run_key(p[0]["id"], p[1], p[2], p[3]) not in RUNS]
    done_t = []
    for i, (cand, model, lam, seed) in enumerate(todo):
        if budget.exceeded():
            log(f"[time cap] {budget.elapsed_h():.2f} h used; stopped with {len(todo) - i} run(s) left in this stage. Re-run the notebook to continue.")
            return False
        t0 = time.time()
        rec = run_fn(cand, model, lam, seed)
        rec["seconds"] = time.time() - t0
        RUNS[run_key(cand["id"], model, lam, seed)] = rec
        if rec["seconds"] > 5:
            done_t.append(rec["seconds"])
            log(f"   {cand['id']:<28s} {model:<9s} lam={lam:<4g} seed={seed}  robust={rec['select_score']:.4f}  cleanF1={rec['val_macro_f1']:.4f}  "
                f"{rec['seconds'] / 60:.1f} min | ~{np.mean(done_t) * (len(todo) - i - 1) / 3600:.2f} h left in this stage")
    return True

def stage1_plan(cands, seed, lam):
    return [(c, m, (0.0 if m == "baseline" else lam), seed) for c in cands for m in ("baseline", "proposed")]

def stage1_table(RUNS, cands, seed, lam):
    rows = []
    for c in cands:
        b, p = RUNS.get(run_key(c["id"], "baseline", 0.0, seed)), RUNS.get(run_key(c["id"], "proposed", lam, seed))
        if b and p:
            rows.append({"id": c["id"], "tag": c["tag"], "lr": c["lr"], "wd": c["wd"], "dropout": c["dropout"], "baseline": b["select_score"],
                         "proposed": p["select_score"], "score": (b["select_score"] + p["select_score"]) / 2,
                         "clean_f1": (b["val_macro_f1"] + p["val_macro_f1"]) / 2})
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

def promote(table, k):
    top = [i for i in table["id"] if i != "default" and table.loc[table["id"] == i, "tag"].iloc[0] != "default"][:k]
    return [table.loc[table["tag"] == "default", "id"].iloc[0]] + top

def stage2_plan(cands_by_id, ids, seeds, lam):
    return [(cands_by_id[i], m, (0.0 if m == "baseline" else lam), s) for i in ids for s in seeds for m in ("baseline", "proposed")]

def seed_scores(RUNS, cid, seeds, lam, col="select_score"):
    return [(RUNS[run_key(cid, "baseline", 0.0, s)][col] + RUNS[run_key(cid, "proposed", lam, s)][col]) / 2 for s in seeds]

def stage2_decide(RUNS, ids, default_id, seeds, lam, min_gain):
    ref = np.array(seed_scores(RUNS, default_id, seeds, lam)); rows = []
    for i in ids:
        sc = np.array(seed_scores(RUNS, i, seeds, lam)); d = sc - ref
        rows.append({"id": i, "mean_score": sc.mean(), "std_over_seeds": sc.std(ddof=1) if len(sc) > 1 else float("nan"),
                     "gain_vs_default": d.mean(), "seeds_better": int((d > 0).sum()), "n_seeds": len(sc),
                     "clean_f1": float(np.mean(seed_scores(RUNS, i, seeds, lam, "val_macro_f1")))})
    tab = pd.DataFrame(rows).sort_values("mean_score", ascending=False).reset_index(drop=True)
    best = tab.iloc[0]
    if best["id"] == default_id:
        return tab, default_id, "the current settings are already the best confirmed candidate"
    if best["gain_vs_default"] >= min_gain and best["seeds_better"] == best["n_seeds"]:
        return tab, best["id"], f"gain {best['gain_vs_default']:+.4f} >= {min_gain} and better on all {best['n_seeds']} fresh seeds"
    return tab, default_id, (f"best candidate {best['id']} gains only {best['gain_vs_default']:+.4f} (needs >= {min_gain} AND all seeds better; "
                             f"better on {best['seeds_better']}/{best['n_seeds']}) -> keep the current settings")

def stage3_plan(cand, lams, seeds):
    return [(cand, "proposed", l, s) for l in lams for s in seeds]

def stage3_decide(RUNS, cid, lams, seeds, lam_default, min_gain):
    base = np.array([RUNS[run_key(cid, "baseline", 0.0, s)]["select_score"] for s in seeds])
    base_c = np.array([RUNS[run_key(cid, "baseline", 0.0, s)]["val_macro_f1"] for s in seeds]); rows = []
    for l in lams:
        p = np.array([RUNS[run_key(cid, "proposed", l, s)]["select_score"] for s in seeds])
        pc = np.array([RUNS[run_key(cid, "proposed", l, s)]["val_macro_f1"] for s in seeds])
        rows.append({"lambda": l, "robust_score": p.mean(), "std_over_seeds": p.std(ddof=1) if len(p) > 1 else float("nan"), "clean_f1": pc.mean(),
                     "robust_minus_baseline": (p - base).mean(), "clean_minus_baseline": (pc - base_c).mean(), "seeds_above_baseline": int((p > base).sum()), "n_seeds": len(p)})
    tab = pd.DataFrame(rows)
    spread = tab["robust_score"].max() - tab["robust_score"].min()
    if spread < min_gain:
        return tab, lam_default, f"the whole lambda grid spans only {spread:.4f} (< {min_gain}) -> lambda is NOT distinguishable; keeping {lam_default:g}"
    best = tab.loc[tab["robust_score"].idxmax(), "lambda"]
    return tab, float(best), f"lambda {best:g} has the highest mean robust score (grid spread {spread:.4f})"
print("Helpers defined.")

In [ ]:
RUNS = {}
BUDGET = Budget(MAX_HOURS)
val_loader = make_loader(dev_ds, batch_size=32, shuffle=False, seed=0)

def run_trial(cand, model, lam, seed):
    fusion = FUSION_BASE if model == "baseline" else FUSION_REL
    kw = model_kwargs_for(fusion, 0.0 if model == "baseline" else lam)
    kw["dropout"] = float(cand["dropout"])
    hp = {**HP, "lr": float(cand["lr"]), "weight_decay": float(cand["wd"])}
    name = "baseline" if model == "baseline" else f"proposed_lam{lam:g}"
    run_dir = os.path.join(TUNE_DIR, cand["id"], name, f"seed{seed}")
    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, seed=seed)
    res = train_run(run_dir, kw, hp, seed, train_loader, val_loader, class_weights, device, EMOTIONS, calib_loader=calib_loader,
                    feature_std=FEATURE_STD, robust_loaders=ROBUST_LOADERS, verbose=False)
    return {"cand_id": cand["id"], "model": model, "lam": float(lam), "seed": int(seed), "select_score": float(res["best_select_score"]),
            "val_macro_f1": float(res["best_val_macro_f1"]), "val_accuracy": float(res["best_val_accuracy"]), "best_epoch": int(res["best_epoch"]),
            "epochs_run": int(res["epochs_run"]), "run_dir": run_dir}

def save_runs():
    pd.DataFrame(list(RUNS.values())).to_csv(os.path.join(TUNE_RES, "runs.csv"), index=False)

DEFAULT = {"lr": float(HP["lr"]), "wd": float(HP["weight_decay"]), "dropout": float(MODEL_KWARGS["dropout"])}
CANDS = sample_candidates(N_RANDOM, TUNE_SEED, DEFAULT, SPACE)
BY_ID = {c["id"]: c for c in CANDS}
pd.DataFrame(CANDS).to_csv(os.path.join(TUNE_RES, "candidates.csv"), index=False)
n1 = len(CANDS) * 2; n2 = (TOP_K + 1) * len(FRESH_SEEDS) * 2; n3 = len(set(LAMBDA_GRID) | {LAMBDA_SCREEN}) * len(FRESH_SEEDS)
print(f"{len(CANDS)} candidates (incl. the reference). Planned runs: stage 1 = {n1}, stage 2 <= {n2}, stage 3 <= {n3} (runs that already exist are skipped). Total <= {n1 + n2 + n3}.")
print(pd.DataFrame(CANDS)[["id", "tag", "lr", "wd", "dropout"]].to_string(index=False))

In [ ]:
ok1 = execute(stage1_plan(CANDS, SCREEN_SEED, LAMBDA_SCREEN), run_trial, RUNS, BUDGET); save_runs()
T1 = stage1_table(RUNS, CANDS, SCREEN_SEED, LAMBDA_SCREEN)
T1.to_csv(os.path.join(TUNE_RES, "stage1_screening.csv"), index=False)
print(T1.round(4).to_string(index=False))
assert ok1, "Stage 1 is not finished (time cap). Re-run this notebook to continue."
IDS2 = promote(T1, TOP_K)
print(f"\nPromoted to confirmation: {IDS2}  (the first entry is the reference = current settings)")

In [ ]:
ok2 = execute(stage2_plan(BY_ID, IDS2, FRESH_SEEDS, LAMBDA_SCREEN), run_trial, RUNS, BUDGET); save_runs()
assert ok2, "Stage 2 is not finished (time cap). Re-run this notebook to continue."
T2, WINNER_ID, WHY2 = stage2_decide(RUNS, IDS2, IDS2[0], FRESH_SEEDS, LAMBDA_SCREEN, MIN_GAIN)
T2.to_csv(os.path.join(TUNE_RES, "stage2_confirmation.csv"), index=False)
print(T2.round(4).to_string(index=False))
print(f"\nDECISION (stage 2): winner = {WINNER_ID}  -- {WHY2}")

In [ ]:
WIN = BY_ID[WINNER_ID]
lams = sorted(set(LAMBDA_GRID) | {LAMBDA_SCREEN})                
ok3 = execute(stage3_plan(WIN, lams, FRESH_SEEDS), run_trial, RUNS, BUDGET); save_runs()
assert ok3, "Stage 3 is not finished (time cap). Re-run this notebook to continue."
T3, LAM_STAR, WHY3 = stage3_decide(RUNS, WINNER_ID, lams, FRESH_SEEDS, LAMBDA_SCREEN, MIN_GAIN)
T3.to_csv(os.path.join(TUNE_RES, "stage3_lambda.csv"), index=False)
print(T3.round(4).to_string(index=False))
print(f"\nDECISION (stage 3): lambda = {LAM_STAR:g}  -- {WHY3}")

In [ ]:
row = T3[T3["lambda"] == LAM_STAR].iloc[0]
env_final = {"MELD_P9_LR": f"{WIN['lr']:.3g}", "MELD_P9_WD": f"{WIN['wd']:.2g}", "MELD_P9_DROPOUT": f"{WIN['dropout']:g}", "MELD_P9_LAMBDAS": f"{LAM_STAR:g}"}
if WINNER_ID == IDS2[0]:                                        
    env_final = {"MELD_P9_LAMBDAS": f"{LAM_STAR:g}"}
recipe = {"winner_recipe": {k: WIN[k] for k in ("id", "lr", "wd", "dropout", "tag")}, "lambda": LAM_STAR, "env_for_final_run": env_final,
          "why_recipe": WHY2, "why_lambda": WHY3, "criterion": "mean robust validation score (Phase 9 rule, rho=%g), validation data only" % ROBUST_RHO,
          "rules": {"min_gain": MIN_GAIN, "screen_seed": SCREEN_SEED, "decision_seeds": FRESH_SEEDS, "n_random": N_RANDOM, "top_k": TOP_K, "search_seed": TUNE_SEED, "space": SPACE},
          "base_recipe_env": {k: v for k, v in os.environ.items() if k.startswith("MELD_P9_")},
          "baseline_vs_proposed_at_winner": {"robust_diff_mean": float(row["robust_minus_baseline"]), "clean_f1_diff_mean": float(row["clean_minus_baseline"]),
                                             "seeds_proposed_above_baseline": f"{int(row['seeds_above_baseline'])}/{int(row['n_seeds'])}"}}
json.dump(recipe, open(os.path.join(TUNE_RES, "recipe.json"), "w"), indent=2)
print(json.dumps(recipe, indent=2))
print("\n=== Use these for the final run (set them BEFORE running Main / Phase 9; keep your Step A / Step B variables as well) ===")
for k, v in env_final.items():
    print(f"{k}={v}")
print("\nNOTE: the tuning seeds are validation-only. The thesis comparison must come from the TEST evaluation (Phase 10, 5 seeds), run once, with these settings for BOTH models.")
print(f"Baseline vs proposed at this recipe on the fresh VALIDATION seeds: robust {row['robust_minus_baseline']:+.4f}, clean macro-F1 {row['clean_minus_baseline']:+.4f}, "
      f"proposed above baseline on {int(row['seeds_above_baseline'])}/{int(row['n_seeds'])} seeds (validation evidence, not a result).")